# T8 It4: the pre-registered finalists, scored once on official TEST and the blind NUDT 707 draw

Everything scored here was written to `results/preregistered_t8.json` (It3) before this notebook was run: the rules, their numbers, the
attitude model and the success criterion. Run once; results are saved, not tuned. The judge (T7 architecture) is the NUDT-only one, as the
team's review asks: trained on the five NUDT design sets, cut chosen on them.

In [1]:
import os, sys, time, json, hashlib
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
from train.checks.candidate_judge import FEATURES_CTX, baseline, decide, judge_probability, paired_gain, summary
from train.checks.holdout import NUDT_DESIGN, answer_change, attitude_frame, bar_frontier, load_t8_table, versus_frontier, with_attitude
from train.checks.two_tier import decide_two_tier
RES = ROOT / "tests/Test_Iteration_8/results"
PRE_PATH = RES / "preregistered_t8.json"
print("pre-registration", PRE_PATH.stat().st_mtime, "sha256", hashlib.sha256(PRE_PATH.read_bytes()).hexdigest())
PRE = json.loads(PRE_PATH.read_text())
FINAL = PRE["final_sets"]
table = load_t8_table(ROOT, locked=False, blind=True)      # the only notebook that loads official_test and nudt707
table.groupby("data").photo.nunique()

pre-registration 1794601023.2584603 sha256 3911f559238b4e20b324cb8fb2b3a087108283725f2f1645ecd2677865315957


data
nudt101          300
nudt202          200
nudt404          200
nudt505          300
nudt606          300
nudt707          300
official_test    194
official_try     185
Name: photo, dtype: int64

In [2]:
t0 = time.time()
pj = judge_probability(table, NUDT_DESIGN, FEATURES_CTX, "gbm")[0]          # trained on NUDT design sets; plain predictions on the final sets
J = lambda d: dict(override=d["override"], min_choice=d["min_choice"], wide_tau=d["wide_tau"])
pc, jet = PRE["pc"], PRE["jetson"]
R = {("pc", "base: bar 8 / 11"): baseline(table, 8, 11, min_choice=3),
     ("pc", "T7 judge, NUDT only"): decide(table, pj, pc["judge_nudt_only"]["tau"], **J(pc["judge_nudt_only"])),
     ("pc", "two-tier"): decide_two_tier(table, pc["two_tier"], "pc"),
     ("pc", "two-tier, a_low 4"): decide_two_tier(table, pc["two_tier_a4"], "pc"),
     ("jetson", "base: Jetson today"): baseline(table, 8, 8, jetson=True, m=5),
     ("jetson", "T7 judge, NUDT only"): decide(table, pj, jet["judge_nudt_only"]["tau"], jetson=True, m=5, **J(jet["judge_nudt_only"])),
     ("jetson", "T7 judge, NUDT only + early exit"): decide(table, pj, jet["judge_nudt_only"]["tau"], jetson=True, m=5, early_tau=0.98, **J(jet["judge_nudt_only"])),
     ("jetson", "two-tier"): decide_two_tier(table, jet["two_tier"], "jetson"),
     ("jetson", "two-tier + early exit"): decide_two_tier(table, jet["two_tier_early"], "jetson")}
R = {k: v[v.data.isin(FINAL)].reset_index(drop=True) for k, v in R.items()}
print("%.0f s" % (time.time() - t0))

14 s


## 1. Decision rules

In [3]:
final_tab = table[table.data.isin(FINAL)].reset_index(drop=True)
FRONT = bar_frontier(final_tab); FRONT.to_csv(RES / "it4_bar_frontier_final.csv", index=False)
rows = []
for (var, name), r in R.items():
    base = R[(var, "base: bar 8 / 11" if var == "pc" else "base: Jetson today")]
    g = paired_gain(base, r, FINAL)
    v = versus_frontier(r, FRONT, var).set_index("data"); a = answer_change(base, r).set_index("data")
    for d in FINAL:
        rows.append(dict(var=var, method=name, data=d, score=v.loc[d, "score"], wrong=v.loc[d, "wrong"], answered=v.loc[d, "n_answered"], queries=v.loc[d, "queries"],
                         gain=g[d], above_bar=v.loc[d, "above_bar"], same_n_bar=v.loc[d, "same_n_bar"], bar_wrong=v.loc[d, "bar_wrong"],
                         added_right=a.loc[d, "added_right"], added_wrong=a.loc[d, "added_wrong"], removed=a.loc[d, "removed"], removed_wrong=a.loc[d, "removed_wrong"]))
OUT = pd.DataFrame(rows); OUT.to_csv(RES / "it4_final.csv", index=False)
pd.concat({k: v for k, v in R.items()}, names=["var", "method"]).reset_index(level=[0, 1]).to_csv(RES / "it4_final_photos.csv", index=False)
display(OUT.set_index(["data", "var", "method"]).sort_index(level=0, sort_remaining=False).round(2))

score  wrong  answered  queries                  gain  above_bar  same_n_bar  bar_wrong  added_right  added_wrong  removed  removed_wrong
data          var    method                                                                                                                                                                     
nudt707       pc     base: bar 8 / 11                  54.04      3       280     7.82  +0.00 [+0.00, +0.00]       0.00           8          3            0            0        0              0
                     T7 judge, NUDT only               55.44      0       278     7.96  +1.40 [+0.20, +3.00]       1.40           8          3            1            0        3              3
                     two-tier                          54.64      1       277     8.10  +0.60 [-1.00, +2.40]       0.60           8          3            1            1        5              3
                     two-tier, a_low 4                 53.84      3       279     8.03  -0.20 [-2.20, +1.80]      -0.20           8          3            1            3        5              3
              jetson base: Jetson today                54.04      2       277     6.33  +0.00 [+0.00, +0.00]       0.00           8          2            0            0        0              0
                     T7 judge, NUDT only               54.84      0       275     6.35  +0.80 [+0.00, +2.00]       1.40           9          2            0            0        2              2
                     T7 judge, NUDT only + early exit  54.84      0       275     2.03  +0.80 [+0.00, +2.00]       1.40           9          2            0            0        2              2
                     two-tier                          53.84      2       276     6.34  -0.20 [-0.60, +0.00]      -0.20           8          2            0            0        1              0
                     two-tier + early exit             53.84      2       276     2.49  -0.20 [-0.60, +0.00]      -0.20           8          2            0            0        1              0
official_test pc     base: bar 8 / 11                  48.62      0       160    10.01  +0.00 [+0.00, +0.00]       0.00           8          0            0            0        0              0
                     T7 judge, NUDT only               53.75      0       177     8.49  +5.13 [+2.97, +7.61]       0.31           6          1           17            0        0              0
                     two-tier                          52.39      0       173     9.14  +3.77 [+1.11, +6.62]       1.61           7          1           18            0        5              0
                     two-tier, a_low 4                 53.01      0       175     8.71  +4.39 [+1.55, +7.30]      -0.43           6          1           20            0        5              0
              jetson base: Jetson today                49.24      0       162     6.74  +0.00 [+0.00, +0.00]       0.00           8          0            0            0        0              0
                     T7 judge, NUDT only               52.82      0       174     6.47  +3.59 [+1.48, +6.06]       1.11           7          1           13            0        1              0
                     T7 judge, NUDT only + early exit  52.70      0       174     2.78  +3.46 [+1.24, +6.00]       0.99           7          1           13            0        1              0
                     two-tier                          51.16      0       169     6.53  +1.92 [-0.31, +4.27]      -0.56           7          1           11            0        4              0
                     two-tier + early exit             50.78      0       169     3.71  +1.55 [-0.80, +4.02]      -0.93           7          1           11            0        4              0

## 2. Attitude correction on official TEST (model fitted on TRY in It3, applied unchanged)

In [4]:
ATT = dict(terms=tuple(PRE["attitude"]["terms"]), coef=np.array(PRE["attitude"]["coef"]), mirror=PRE["attitude"]["mirror"])
DUMP = str(ROOT / "tests/Test_Iteration_6/results/rich_dump/official")
rows = []
for (var, name), r in R.items():
    off = r[r.data == "official_test"]
    fixed = with_attitude(off, attitude_frame(off, DUMP), ATT)
    g = paired_gain(off, fixed, ["official_test"])["official_test"]
    ans = off.stage != "skip"
    rows.append(dict(var=var, method=name, score=6 * off.pts.mean(), with_attitude=6 * fixed.pts.mean(), gain=g,
                     within1_before=(off.err_sigma[ans] <= 1).mean(), within1_after=(fixed.err_sigma[ans] <= 1).mean(), wrong_after=int((fixed.pts < 0).sum())))
ATT_OUT = pd.DataFrame(rows); ATT_OUT.to_csv(RES / "it4_attitude_test.csv", index=False); display(ATT_OUT.round(3))

,var,method,score,with_attitude,gain,within1_before,within1_after,wrong_after
0,pc,base: bar 8 / 11,48.619,48.742,"+0.12 [-0.37, +0.62]",0.956,0.962,0
1,pc,"T7 judge, NUDT only",53.753,53.876,"+0.12 [-0.37, +0.62]",0.955,0.960,0
2,pc,two-tier,52.392,52.639,"+0.25 [-0.37, +0.87]",0.948,0.960,0
3,pc,"two-tier, a_low 4",53.010,53.258,"+0.25 [-0.37, +0.87]",0.949,0.960,0
4,jetson,base: Jetson today,49.237,49.361,"+0.12 [-0.37, +0.62]",0.957,0.963,0
5,jetson,"T7 judge, NUDT only",52.825,52.948,"+0.12 [-0.37, +0.62]",0.954,0.960,0
6,jetson,"T7 judge, NUDT only + early exit",52.701,53.072,"+0.37 [-0.37, +1.24]",0.948,0.966,0
7,jetson,two-tier,51.155,51.402,"+0.25 [-0.37, +0.87]",0.947,0.959,0
8,jetson,two-tier + early exit,50.784,51.402,"+0.62 [-0.12, +1.36]",0.929,0.959,0


## 3. Against the pre-registered success criterion

In [5]:
def verdict(var, name):
    base = OUT[(OUT["var"] == var) & OUT.method.str.startswith("base")].set_index("data")
    m = OUT[(OUT["var"] == var) & (OUT.method == name)].set_index("data")
    lo = {d: float(m.loc[d, "gain"].split("[")[1].split(",")[0]) for d in FINAL}
    mean = {d: float(m.loc[d, "gain"].split(" ")[0]) for d in FINAL}
    safe = all(m.loc[d, "wrong"] <= base.loc[d, "wrong"] for d in FINAL)
    return dict(var=var, method=name, no_more_wrong=safe, gain_nonneg=all(mean[d] >= 0 for d in FINAL), interval_above_0=any(lo[d] > 0 for d in FINAL),
                counts=safe and all(mean[d] >= 0 for d in FINAL) and any(lo[d] > 0 for d in FINAL))
V = pd.DataFrame([verdict(v, n) for (v, n) in R if not n.startswith("base")]); V.to_csv(RES / "it4_verdict.csv", index=False); V

,var,method,no_more_wrong,gain_nonneg,interval_above_0,counts
0,pc,"T7 judge, NUDT only",True,True,True,True
1,pc,two-tier,True,True,True,True
2,pc,"two-tier, a_low 4",True,False,True,False
3,jetson,"T7 judge, NUDT only",True,True,True,True
4,jetson,"T7 judge, NUDT only + early exit",True,True,True,True
5,jetson,two-tier,True,False,False,False
6,jetson,two-tier + early exit,True,False,False,False
